In [8]:
import torch
from PIL import Image
from torchvision import transforms
import numpy as np
from model.aesthetic_regressor import AestheticRegressor

In [ ]:
def load_trained_model(weight_path, device, n_factors=5, activation='tanh'):
    model = AestheticRegressor(
        n_factors=n_factors,
        activation=activation,
    ).to(device)
    #load trọng số
    state_dict = torch.load(weight_path, map_location=device)
    model.load_state_dict(state_dict, strict=False)
    model.eval()
    return model

In [24]:
def predict_image_adjustments(image_path, model, device=None):

    device = device or ('cuda:0' if torch.cuda.is_available() else 'cpu')

    factors = ['saturation', 'brightness', 'tint', 'temperature', 'contrast']
    factors_coefs = {
        'saturation': 43,
        'brightness': 43,
        'tint': 30,
        'temperature': 30,
        'contrast': 43,
    }
    transform = transforms.Compose([
        transforms.Resize((640, 640)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])

    class Normalizer:
        def __init__(self, coefs):
            self.coefs = np.array(list(coefs.values()), dtype=np.float32)
        def transform(self, labels):
            return labels / self.coefs
        def inverse_transform(self, norm_labels):
            return norm_labels * self.coefs

    label_normalizer = Normalizer(factors_coefs)

    image = Image.open(image_path).convert("RGB")
    image_tensor = transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        preds = model(image_tensor).cpu().numpy().flatten()

    preds_denorm = label_normalizer.inverse_transform(preds)
    result = {factors[i]: float(preds_denorm[i]) for i in range(len(factors))}
    return result

In [25]:
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'

model = load_trained_model(
    weight_path=r'C:\MINE\NCKH\chat2edit-framework\image-editing\image_enhancing\model\epoch_25.pth',
    device=device,
    activation='tanh',
    backbone='resnet18'
)
image_path = r'C:\MINE\NCKH\chat2edit-framework\image-editing\test_image\pic27.jpg'
results = predict_image_adjustments(image_path, model, device)

print("Kết quả dự đoán chỉnh sửa ảnh:")
for k, v in results.items():
    print(f"  {k}: {v:.2f}")


Initializing resnet18 model...


c:\Users\DELL\AppData\Local\Programs\Python\Python311\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\DELL\AppData\Local\Programs\Python\Python311\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Number of trainable parameters: 11689512
Kết quả dự đoán chỉnh sửa ảnh:
  saturation: -28.42
  brightness: 24.44
  tint: 7.51
  temperature: -17.82
  contrast: 10.71
